In [46]:
import os
from dotenv import load_dotenv

load_dotenv()

if os.environ.get("OPENAI_API_KEY"):
    print("Bro API Key Exists")
else:
    raise ValueError("OPENAI_API_KEY not found")

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnableLambda
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableParallel

Bro API Key Exists


# Chain with Parallel Chains

In [47]:
# TASK -1[Prompt]


prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are Movie Summarizer"),
    ("human","Please summarize the moview in 50 letters: {input}")
])

In [48]:

# TASK-2 [LLM]
llm_openai = ChatOpenAI(model='gpt-5-mini', temperature=0)



In [49]:
# TASK-3 String parser
str_parser = StrOutputParser()

In [50]:
# TASK - 4 - Custom Runnable dict
def dictionary_maker(text:str) -> dict:
    return {"text": text}

dictionary_maker_runnable = RunnableLambda(dictionary_maker) 

# Parallel Chain - 1

In [51]:
# TASK -1 [Promt]

linkedin_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a Linkedin Post Generator"),
    ("human", " Create a post for the follwing text for Linkedin: {text}")
])

# TASK-2 [llm]

llm_openai = ChatOpenAI(model="gpt-5-mini", temperature=0)

#TASK-3 [str output]

str_parser = StrOutputParser()

chain_linekdin = linkedin_prompt | llm_openai | str_parser


# Parallel Chain-2

In [52]:

def insta_chain(text:dict):
    text = text["text"]
# TASK -1 [prompt]
    insta_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a Instagram Post Generator"),
    ("human", " Create a post for the follwing text for Instagram: {text}")
    ])

    # TASK-2 [llm]
    
    llm_openai = ChatOpenAI(model="gpt-5-mini", temperature=0)
    
    #TASK-3 [str output]
    
    str_parser = StrOutputParser()
    
    chain_insta = insta_prompt | llm_openai | str_parser

    result = chain_insta.invoke(text)
    return result

insta_chain_runnable = RunnableLambda(insta_chain)

# Final Orchestration

In [53]:
final_chain = (
        prompt_template | 
        llm_openai | 
        str_parser | 
        dictionary_maker_runnable |
        RunnableParallel(branches={"linkedin":chain_linekdin, "instagram":insta_chain_runnable})
)

In [54]:
#final_output = final_chain.invoke("Bahubali")

# Chain as a Runnable

In [55]:
#TASK -1 [Beautify]
def beautify(final_response: dict) -> dict:
    linkedin_response = final_response['branches']['linkedin']
    instagram_response = final_response['branches']['instagram']

    return {"linkedin": linkedin_response, "instagram": instagram_response}

beautify_runnable = RunnableLambda(beautify)

# TASK-2 [Final Chain]

# final_chain

# beautified chain

beautified_chain = final_chain | beautify_runnable

beautified_chain.invoke("Pushpa")

{'linkedin': 'Option 1 — Punchy hook + lesson\n"Ruthless smuggler rises, fights, claims power. Now."\n\nA powerful image — but what does it teach leaders? Speed and aggression can win battles, but without values, trust and long-term strategy, power is brittle. If you want influence that lasts, pair audacity with ethics, systems and people who will follow you when the stakes are highest.\n\nWhat’s an example you’ve seen where ambition outpaced integrity — and what happened next?\n#Leadership #Ethics #Strategy\n\nOption 2 — Reflective leadership post\n"Ruthless smuggler rises, fights, claims power. Now."\n\nStories like this force a question: do we celebrate results or the way they’re achieved? In business, short-term wins earned through ruthless tactics can damage reputation, teams and sustainability. True leadership is winning with purpose — aligned values, accountability and a plan to retain trust after the victory.\n\nHow do you balance bold moves with ethical guardrails in your orga